# Day 4 v2 — Model 06: AITeamVN/Vietnamese_Embedding Fine-tune (top-4 layers)

**Architecture:** `AITeamVN/Vietnamese_Embedding` (partial unfreeze: top 4/24 layers) → mean_pooling → price head + aux category head.

**Techniques:** LLRD + EMA (0.999) + Huber Loss + AMP + Cosine Warmup + Multi-task aux head.

**vs Model 05 (frozen encoder, MAE=76.1k):** Fine-tuning top layers should adapt VN embeddings to price-predictive features.

**Target:** MAE < 65k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.bert_finetune_model import BERTFinetuneRunner

MODEL_NAME = "AITeamVN/Vietnamese_Embedding"

print(f"Model: {MODEL_NAME}")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Runner

- Tokenize 269K train + 3926 val (stored as tensors, ~550MB for max_length=256)
- Freeze bottom 20/24 transformer layers, unfreeze top 4
- Configure LLRD param groups: head lr=2e-5, each lower layer × decay=0.9
- Approx trainable params: ~15M encoder + ~1M heads = ~16M total

In [ ]:
runner = BERTFinetuneRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=4,
    batch_size=32,
    max_length=256,
    base_lr=2e-5,
    weight_decay=0.02,
    llrd_decay=0.9,
    dropout=0.2,
)

## 3. Train

10 epochs, early stopping patience=3. Val MAE evaluated on full 3926 samples per epoch using EMA model.

Expected: ~30-40 min/epoch on RTX 3090 Ti (269K × 256 tokens).

In [ ]:
history = runner.train(
    epochs=10,
    patience=3,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=0.999,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
)

## 4. Training History

In [ ]:
plot_training_history(history, title="AITeamVN Fine-tune top-4 layers")

## 5. Save Weights + Val Predictions + Test Predictions

In [ ]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/aitvn_finetune.pth")
print("Saved weights/aitvn_finetune.pth")

Path("val_predictions").mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/aitvn_finetune_val.json", "w") as f:
    json.dump(val_preds, f)

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open("val_predictions/aitvn_finetune_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

## 6. Evaluate on 200 Test Samples

In [ ]:
def aitvn_finetune_pricer(item):
    return runner.inference(item)

results = evaluate(aitvn_finetune_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

## 7. Sanity Check — Load Roundtrip

In [ ]:

# Verify checkpoint keys and one sample inference
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load("weights/aitvn_finetune.pth", map_location="cpu", weights_only=False)
expected_keys = {"ema_state_dict", "y_mean", "y_std", "model_name", "keep_top_layers", "cat_classes"}
assert expected_keys == set(ckpt.keys()), f"Unexpected keys: {set(ckpt.keys())}"
print(f"\nCheckpoint keys OK: {sorted(ckpt.keys())}")
print(f"y_mean={ckpt['y_mean']:.4f} | y_std={ckpt['y_std']:.4f} | keep_top_layers={ckpt['keep_top_layers']}")
print(f"Categories ({len(ckpt['cat_classes'])}): {ckpt['cat_classes']}")
